In [ ]:

import pandas as pd
import plotly.graph_objects as go
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

In [ ]:
# set target positions for a spread of 25, 50, 75, 100, etc...

# get data

In [ ]:
DAY = 0
df = pd.read_csv(f"./round-3-island-data-bottle/prices_round_3_day_{DAY}.csv", sep=";", header=0)

In [ ]:
def swmid(row):
    bid = row['bid_price_1']
    ask = row['ask_price_1']
    bid_volume = row['bid_volume_1']
    ask_volume = row['ask_volume_1']
    return ((bid*ask_volume + ask*bid_volume)/(bid_volume + ask_volume))

def mm_mid_basket(row, volume_cutoff=10):
    # Find the best bid with volume >= volume_cutoff
    for i in range(1, 4):
        if row[f'bid_volume_{i}'] >= volume_cutoff:
            best_bid = row[f'bid_price_{i}']
            break
    else:
        # No bid with sufficient volume found
        best_bid = None

    # Find the best ask with volume >= volume_cutoff
    for i in range(1, 4):
        if row[f'ask_volume_{i}'] >= volume_cutoff:
            best_ask = row[f'ask_price_{i}']
            break
    else:
        # No ask with sufficient volume found
        best_ask = None

    # Calculate the mid-price if both best bid and best ask are found
    if best_bid is not None and best_ask is not None:
        mid_price = (best_bid + best_ask) / 2
        return mid_price
    else:
        # Return the mid_price column value as default
        return row['mid_price']
    
    

def fair_price(row):
    if row['product'] == 'GIFT_BASKET':
        return mm_mid_basket(row, volume_cutoff=10)
    else:
        return swmid(row)


In [ ]:

df['fair'] = df.apply(fair_price, axis=1)

In [ ]:
# Define the weights dictionary
weights = {
    'CHOCOLATE': 4,
    'STRAWBERRIES': 6,
    'ROSES': 1
}

# Get the unique product names
products = df['product'].unique()

# Create a new dataframe 'df_fairs' with the desired columns
columns = ['timestamp'] + list(products) + ['SYNTHETIC']
df_fairs = pd.DataFrame(columns=columns)

# Iterate over unique timestamps in the original dataframe
for timestamp in df['timestamp'].unique():
    # Get the rows for the current timestamp
    rows = df[df['timestamp'] == timestamp]
    
    # Create a dictionary to store the fair values for each product
    fairs = {}
    
    # Iterate over each product and extract its fair value
    for product in products:
        fair = rows.loc[rows['product'] == product, 'fair'].values[0]
        fairs[product] = fair
    
    # Calculate the synthetic fair
    synthetic_fair = sum(fairs[product] * weights.get(product, 0) for product in ['CHOCOLATE', 'STRAWBERRIES', 'ROSES'])
    
    # Create a new row as a DataFrame
    new_row = pd.DataFrame({'timestamp': [timestamp], **{product: [fairs[product]] for product in products}, 'SYNTHETIC': [synthetic_fair]})
    
    # Concatenate the new row with df_fairs
    df_fairs = pd.concat([df_fairs, new_row], ignore_index=True)

# Reset the index of df_fairs (optional)
df_fairs = df_fairs.reset_index(drop=True)

In [ ]:
df = df_fairs

# binning target positions

In [ ]:
df_fairs = df.copy()

In [ ]:
df_fairs['SPREAD'] = df_fairs['GIFT_BASKET'] - df_fairs['SYNTHETIC'] - 370

In [ ]:
df_fairs

In [ ]:
bins = [-np.inf, -150, -100,-75, -50,-25, 25, 50, 75, 100, 150, np.inf]
labels = [60,45, 30, 20, 10, 0, -10, -20, -30,-45, -60]

# Create the "POSITION" column based on the conditions
df_fairs['POSITION'] = pd.cut(df_fairs['SPREAD'], bins=bins, labels=labels, right=False)

# Convert the "POSITION" column to numeric type
df_fairs['POSITION'] = pd.to_numeric(df_fairs['POSITION'])

In [ ]:
df_fairs

In [ ]:
df_fairs['POSITION_DIFF'] = df_fairs['POSITION'].diff()

# Create a new column "TRADE" to mark the trades
df_fairs['TRADE'] = np.where(df_fairs['POSITION_DIFF'] != 0, df_fairs['POSITION_DIFF'], 0)

# Create a new column "CASH_CHANGE" to calculate the cash change for each trade
df_fairs['CASH_CHANGE'] = -df_fairs['TRADE'] * df_fairs['SPREAD']

# Create a new column "CASH" to track the cumulative cash position
df_fairs['CASH'] = df_fairs['CASH_CHANGE'].cumsum()

In [ ]:
df_fairs

In [ ]:
# lagged returns from n timesteps negative when spread is positive -> sell signal
# lagged returns from n timesteps positive when spread is negative -> buy signal

# trading with ewma fair

In [ ]:
df_fairs = df.copy()

In [ ]:
df_fairs

In [ ]:
df_fairs['BASKET_MINUS_SYNTHETIC'] = df_fairs['GIFT_BASKET'] - df_fairs["SYNTHETIC"]

In [ ]:
alpha = 0.1
df_fairs['BASKET_MINUS_SYNTHETIC_EMA'] = df_fairs['BASKET_MINUS_SYNTHETIC'].ewm(alpha=alpha, adjust=False).mean()
df_fairs['BASKET_MINUS_SYNTHETIC_MINUS_EMA'] = df_fairs['BASKET_MINUS_SYNTHETIC'] - df_fairs['BASKET_MINUS_SYNTHETIC_EMA']

In [ ]:
df_fairs